# 06 — Upcoming Fixture Predictions

A notebook preview of what `app.py`'s Upcoming Fixtures and Value Bets tabs render: this gameweek's fixtures, model predictions across every market, and (if `ODDS_API_KEY` is set) live-odds value bets.

In [1]:
import sys
sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')

import pandas as pd
from pl_predictor.data.fixtures import get_upcoming_fixtures
from pl_predictor.data.odds_api import fetch_epl_odds, OddsAPIKeyMissing
from pl_predictor.models.manifest import load_models
from pl_predictor.odds.value_bets import build_value_bet_table

fixtures = get_upcoming_fixtures()
models = load_models()
print(f'{len(fixtures)} upcoming fixtures')
fixtures.head()

14 upcoming fixtures


,event_id,commence_time,team_home,team_away,has_odds
0,f5086c74d4a767e3ad7ea69bd4a66b76,2026-08-23 13:00:00+00:00,Brighton,Aston Villa,True
1,87c7c9bfd5e3072bded20c702f553c63,2026-08-23 13:00:00+00:00,Man City,Bournemouth,True
2,ea0bacc4c23f813c6af97ec9b306a850,2026-08-23 15:30:00+00:00,Newcastle,Liverpool,True
3,4e4a813bf4218cc527e6f8ef2351170d,2026-08-24 19:00:00+00:00,Fulham,Chelsea,True
4,373cc6e2fb57f471cd2deab9a28a6edb,2026-08-28 19:00:00+00:00,Crystal Palace,Man City,True


In [2]:
try:
    odds_df = fetch_epl_odds()
except OddsAPIKeyMissing as e:
    print(e)
    odds_df = pd.DataFrame()

table = build_value_bet_table(fixtures, odds_df, models)
table[['team_home', 'team_away', 'home_win_prob', 'draw_prob', 'away_win_prob', 'top_scoreline', 'btts_yes_prob', 'over_2_5_prob', 'value_bet_flags']]

,team_home,team_away,home_win_prob,draw_prob,away_win_prob,top_scoreline,btts_yes_prob,over_2_5_prob,value_bet_flags
0,Brighton,Aston Villa,0.392658,0.242424,0.364918,1-1,0.624173,0.597096,[away_win]
1,Man City,Bournemouth,0.689062,0.184393,0.126545,2-0,0.527219,0.605440,[under_2_5]
2,Newcastle,Liverpool,0.304135,0.228645,0.467220,1-1,0.652141,0.642617,[home_win]
3,Fulham,Chelsea,0.312683,0.255744,0.431573,1-1,0.566289,0.526881,"[home_win, under_2_5]"
4,Crystal Palace,Man City,0.189090,0.228120,0.582790,0-1,0.529127,0.536167,[]
5,Liverpool,Nott'm Forest,0.667079,0.188335,0.144587,2-0,0.563811,0.628978,[]
6,Bournemouth,Everton,0.425727,0.290686,0.283587,1-0,0.457046,0.391753,[under_2_5]
7,Coventry,Hull,0.370910,0.258180,0.370910,1-1,0.548725,0.506375,[away_win]
8,Tottenham,Newcastle,0.349809,0.221097,0.429094,1-1,0.699279,0.696882,"[away_win, over_2_5]"
9,Leeds,Brentford,0.255629,0.220729,0.523643,1-1,0.641087,0.644414,"[away_win, over_2_5]"


## Value bets

In [3]:
flagged = table[table['value_bet_flags'].apply(len) > 0]
flagged if not flagged.empty else 'No value bets above the edge threshold right now (or no live odds configured).'

,event_id,commence_time,team_home,team_away,home_win_prob,draw_prob,away_win_prob,btts_yes_prob,over_2_5_prob,under_2_5_prob,...,draw_edge,away_win_implied,away_win_edge,over_2_5_implied,over_2_5_edge,under_2_5_implied,under_2_5_edge,value_bet_flags,corners_market_note,cards_market_note
0,f5086c74d4a767e3ad7ea69bd4a66b76,2026-08-23 13:00:00+00:00,Brighton,Aston Villa,0.392658,0.242424,0.364918,0.624173,0.597096,0.402904,...,-0.022826,0.301573,0.063345,0.552602,0.044494,0.447398,-0.044494,[away_win],No live market (Odds API doesn't cover corners),No live market (Odds API doesn't cover cards)
1,87c7c9bfd5e3072bded20c702f553c63,2026-08-23 13:00:00+00:00,Man City,Bournemouth,0.689062,0.184393,0.126545,0.527219,0.605440,0.394560,...,-0.005994,0.154388,-0.027843,0.675325,-0.069885,0.324675,0.069885,[under_2_5],No live market (Odds API doesn't cover corners),No live market (Odds API doesn't cover cards)
2,ea0bacc4c23f813c6af97ec9b306a850,2026-08-23 15:30:00+00:00,Newcastle,Liverpool,0.304135,0.228645,0.467220,0.652141,0.642617,0.357383,...,-0.013982,0.508655,-0.041435,0.633333,0.009283,0.366667,-0.009283,[home_win],No live market (Odds API doesn't cover corners),No live market (Odds API doesn't cover cards)
3,4e4a813bf4218cc527e6f8ef2351170d,2026-08-24 19:00:00+00:00,Fulham,Chelsea,0.312683,0.255744,0.431573,0.566289,0.526881,0.473119,...,0.007653,0.521234,-0.089660,0.582314,-0.055433,0.417686,0.055433,"[home_win, under_2_5]",No live market (Odds API doesn't cover corners),No live market (Odds API doesn't cover cards)
6,ff4818bd73fa63c0a1e4c98e50c7a2e5,2026-08-29 14:00:00+00:00,Bournemouth,Everton,0.425727,0.290686,0.283587,0.457046,0.391753,0.608247,...,0.034477,0.246040,0.037547,0.525253,-0.133499,0.474747,0.133499,[under_2_5],No live market (Odds API doesn't cover corners),No live market (Odds API doesn't cover cards)
7,374dd92785cc06d6514d275515219c44,2026-08-29 14:00:00+00:00,Coventry,Hull,0.370910,0.258180,0.370910,0.548725,0.506375,0.493625,...,0.005173,0.246315,0.124595,0.556022,-0.049647,0.443978,0.049647,[away_win],No live market (Odds API doesn't cover corners),No live market (Odds API doesn't cover cards)
8,7a8746a45dc76378359a7d9cc5474bc8,2026-08-29 16:30:00+00:00,Tottenham,Newcastle,0.349809,0.221097,0.429094,0.699279,0.696882,0.303118,...,-0.031654,0.318673,0.110421,0.545115,0.151767,0.454885,-0.151767,"[away_win, over_2_5]",No live market (Odds API doesn't cover corners),No live market (Odds API doesn't cover cards)
9,e479c08421b8549c6959c2b84f6f3f79,2026-08-30 13:00:00+00:00,Leeds,Brentford,0.255629,0.220729,0.523643,0.641087,0.644414,0.355586,...,-0.056909,0.367295,0.156348,0.508490,0.135924,0.491510,-0.135924,"[away_win, over_2_5]",No live market (Odds API doesn't cover corners),No live market (Odds API doesn't cover cards)
